In [1]:
import pandas as pd

print("Pandas is installed successfully!")
print("Version:", pd.__version__)

Pandas is installed successfully!
Version: 2.3.3


In [2]:
# Import the libraries we need
import requests        # for making API calls
import pandas as pd    # for data manipulation
import time            # to pause between API calls (respect rate limits)
import os              # for file path operations

# Confirm everything imported correctly
print("Libraries loaded successfully")
print(f"pandas version: {pd.__version__}")

Libraries loaded successfully
pandas version: 2.3.3


## CoinGecko API Endpoint

We will use the CoinGecko Markets endpoint to retrieve cryptocurrency market data.

Endpoint:

https://api.coingecko.com/api/v3/coins/markets

Parameters:
- vs_currency: Currency for prices (e.g., usd)
- order: Sort order for results
- per_page: Number of coins to return
- page: Page number
- sparkline: Include sparkline data (true/false)

The endpoint returns:
- Coin name
- Symbol
- Current price
- Market capitalization
- Trading volume
- Price change percentages

In [3]:
#Function Definitio
def get_coin_data(coin_id, days=365):
    """
    Pull daily price and volume data from CoinGecko.
    coin_id: "bitcoin" or "ethereum"
    days: number of days of history to pull
    Returns a pandas DataFrame
    """
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"

    params = {
        "vs_currency": "usd",
        "days": days,
        "interval": "daily"
    }

    # Make the API request
    response = requests.get(url, params=params)

    # Check if the request was successful
    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        return None

    # Parse the JSON response
    data = response.json()

    # Extract prices and volumes
    prices = data["prices"]            # list of [timestamp, price]
    volumes = data["total_volumes"]    # list of [timestamp, volume]

    # Convert to DataFrame
    df_price = pd.DataFrame(prices, columns=["timestamp", f"{coin_id}_price_usd"])
    df_volume = pd.DataFrame(volumes, columns=["timestamp", f"{coin_id}_volume_usd"])

    # Merge price and volume on timestamp
    df = pd.merge(df_price, df_volume, on="timestamp")

    # Convert timestamp from milliseconds to readable date
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
    df = df.drop("timestamp", axis=1)

    # Put date first
    df = df[["date", f"{coin_id}_price_usd", f"{coin_id}_volume_usd"]]

    #Enforce one row per date
    df = df.groupby("date", as_index=False).last()

    return df

print("Function defined. Ready to pull data.")

Function defined. Ready to pull data.


In [4]:
# Test: pull 7 days of Bitcoin data ONLY
df_test = get_coin_data("bitcoin", days=7)

print(df_test)
print(f"\nShape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")

         date  bitcoin_price_usd  bitcoin_volume_usd
0  2026-06-14       64377.576689        1.752280e+10
1  2026-06-15       65713.623560        2.288758e+10
2  2026-06-16       66300.958537        3.391518e+10
3  2026-06-17       65598.943967        2.556770e+10
4  2026-06-18       64422.573665        3.234518e+10
5  2026-06-19       62900.227093        3.127974e+10
6  2026-06-20       63665.351735        1.922880e+10

Shape: (7, 3)
Columns: ['date', 'bitcoin_price_usd', 'bitcoin_volume_usd']


In [5]:
# Pull Bitcoin data — 365 days
print("Pulling Bitcoin data...")
df_btc = get_coin_data("bitcoin", days=365)
print(f"Bitcoin: {df_btc.shape[0]} rows pulled")

# Wait (rate limit safety)
print("Waiting 12 seconds before next call...")
time.sleep(12)

# Pull Ethereum data — 365 days
print("Pulling Ethereum data...")
df_eth = get_coin_data("ethereum", days=365)
print(f"Ethereum: {df_eth.shape[0]} rows pulled")

# Wait again (important — don't skip this)
print("Waiting 12 seconds before next call...")
time.sleep(12)

# Pull USDT data — 365 days
print("Pulling USDT data...")
df_usdt = get_coin_data("tether", days=365)
print(f"USDT: {df_usdt.shape[0]} rows pulled")

# Merge step-by-step (safer than one big merge chain)
df_crypto = pd.merge(df_btc, df_eth, on="date", how="inner")
df_crypto = pd.merge(df_crypto, df_usdt, on="date", how="inner")

# Final checks
print(f"\nMerged dataset shape: {df_crypto.shape}")
print(f"Date range: {df_crypto['date'].min()} to {df_crypto['date'].max()}")

print("\nColumns:")
print(df_crypto.columns.tolist())

print("\nFirst 3 rows:")
print(df_crypto.head(3))

Pulling Bitcoin data...
Bitcoin: 365 rows pulled
Waiting 12 seconds before next call...
Pulling Ethereum data...
Ethereum: 365 rows pulled
Waiting 12 seconds before next call...
Pulling USDT data...
USDT: 365 rows pulled

Merged dataset shape: (365, 7)
Date range: 2025-06-21 to 2026-06-20

Columns:
['date', 'bitcoin_price_usd', 'bitcoin_volume_usd', 'ethereum_price_usd', 'ethereum_volume_usd', 'tether_price_usd', 'tether_volume_usd']

First 3 rows:
         date  bitcoin_price_usd  bitcoin_volume_usd  ethereum_price_usd  \
0  2025-06-21      103290.105145        3.063275e+10         2405.695434   
1  2025-06-22      101532.568385        2.164446e+10         2270.581996   
2  2025-06-23      100852.582646        5.128714e+10         2227.428727   

   ethereum_volume_usd  tether_price_usd  tether_volume_usd  
0         2.048282e+10          1.000017       2.772718e+10  
1         1.615561e+10          1.000177       4.036210e+10  
2         2.818144e+10          1.000176       3.761731e

In [19]:
df_crypto.duplicated(subset=["date"]).sum()

np.int64(0)

In [20]:
df_crypto[df_crypto.duplicated(subset=["date"], keep=False)].sort_values("date")

,date,bitcoin_price_usd,bitcoin_volume_usd,ethereum_price_usd,ethereum_volume_usd,tether_price_usd,tether_volume_usd


In [21]:
#Bitcoin
df_btc = get_coin_data("bitcoin", days=365)
df_btc["date"].is_unique

True

In [22]:
df_ethereum = get_coin_data("ethereum", days=365)
df_ethereum["date"].is_unique

True

In [23]:
# USDT (Tether)
df_usdt = get_coin_data("tether", days=365)
print(df_usdt["date"].is_unique)
print(df_usdt.shape)

True
(365, 3)


In [25]:
# Merge Ethereum + USDT on date (inner join keeps only matching dates)
df_crypto = df_ethereum.merge(df_usdt, on="date", how="inner")

print(df_crypto.shape)
print(df_crypto.columns)

(365, 5)
Index(['date', 'ethereum_price_usd', 'ethereum_volume_usd', 'tether_price_usd',
       'tether_volume_usd'],
      dtype='object')


In [26]:
# Merge BTC
df_btc = get_coin_data("bitcoin", days=365)

print(df_btc.shape)
print(df_btc.columns)
print(df_btc["date"].is_unique)

(365, 3)
Index(['date', 'bitcoin_price_usd', 'bitcoin_volume_usd'], dtype='object')
True


In [27]:
df_btc.shape, df_btc.columns, df_btc["date"].is_unique

((365, 3),
 Index(['date', 'bitcoin_price_usd', 'bitcoin_volume_usd'], dtype='object'),
 True)

In [29]:
# Start with BTC as base (best anchor)
df_crypto = df_btc.merge(df_ethereum, on="date", how="inner")

# Add USDT liquidity layer
df_crypto = df_crypto.merge(df_usdt, on="date", how="inner")

# Final structure check
print("Shape:", df_crypto.shape)
print(df_crypto.columns)
print("Duplicate dates:", df_crypto["date"].duplicated().sum())

Shape: (365, 7)
Index(['date', 'bitcoin_price_usd', 'bitcoin_volume_usd', 'ethereum_price_usd',
       'ethereum_volume_usd', 'tether_price_usd', 'tether_volume_usd'],
      dtype='object')
Duplicate dates: 0


In [30]:
# Total market liquidity proxy (activity in crypto ecosystem)
df_crypto["market_liquidity"] = (
    df_crypto["bitcoin_volume_usd"] +
    df_crypto["ethereum_volume_usd"] +
    df_crypto["tether_volume_usd"]
)

print(df_crypto[["date", "market_liquidity"]].head())

         date  market_liquidity
0  2025-06-21      7.884275e+10
1  2025-06-22      7.816217e+10
2  2025-06-23      1.170859e+11
3  2025-06-24      1.061338e+11
4  2025-06-25      8.524248e+10


In [31]:
df_crypto["liquidity_index"] = (
    df_crypto["market_liquidity"] /
    df_crypto["market_liquidity"].rolling(7, min_periods=1).mean()
)

print(df_crypto[["date", "liquidity_index"]].head(10))

         date  liquidity_index
0  2025-06-21         1.000000
1  2025-06-22         0.995665
2  2025-06-23         1.281538
3  2025-06-24         1.116538
4  2025-06-25         0.915666
5  2025-06-26         1.001597
6  2025-06-27         0.821224
7  2025-06-28         0.700097
8  2025-06-29         0.432993
9  2025-06-30         0.628576


In [32]:
# 1. Recompute returns (daily market movement)
df_crypto["btc_return"] = df_crypto["bitcoin_price_usd"].pct_change()
df_crypto["eth_return"] = df_crypto["ethereum_price_usd"].pct_change()

# Fill first NaN from returns
df_crypto["btc_return"] = df_crypto["btc_return"].fillna(0)
df_crypto["eth_return"] = df_crypto["eth_return"].fillna(0)

# 2. Smooth signals (light smoothing)
df_crypto["btc_signal"] = df_crypto["btc_return"].rolling(3, min_periods=1).mean()
df_crypto["eth_signal"] = df_crypto["eth_return"].rolling(3, min_periods=1).mean()

# 3. Raw Crypto Market Index (CMI)
df_crypto["CMI"] = (
    0.45 * df_crypto["btc_signal"] +
    0.35 * df_crypto["eth_signal"] +
    0.20 * df_crypto["liquidity_index"]
)

# 4. Handle missing values
df_crypto["CMI"] = df_crypto["CMI"].bfill()

# 5. Normalize to 0–100 scale (IMPORTANT for MEPS later)
df_crypto["CMI_index"] = (
    (df_crypto["CMI"] - df_crypto["CMI"].min()) /
    (df_crypto["CMI"].max() - df_crypto["CMI"].min())
) * 100

# 6. Final preview
print(df_crypto[["date", "CMI", "CMI_index"]].head(10))

         date       CMI  CMI_index
0  2025-06-21  0.200000  34.431708
1  2025-06-22  0.185476  30.723694
2  2025-06-23  0.243981  45.660016
3  2025-06-24  0.228201  41.631404
4  2025-06-25  0.198881  34.146131
5  2025-06-26  0.219679  39.455651
6  2025-06-27  0.165922  25.731574
7  2025-06-28  0.140466  19.232692
8  2025-06-29  0.087689   5.758724
9  2025-06-30  0.131893  17.044037


In [34]:
df_crypto["btc_return"] = df_crypto["bitcoin_price_usd"].pct_change().fillna(0)
df_crypto["eth_return"] = df_crypto["ethereum_price_usd"].pct_change().fillna(0)

df_crypto["btc_signal"] = df_crypto["btc_return"].rolling(3, min_periods=1).mean()
df_crypto["eth_signal"] = df_crypto["eth_return"].rolling(3, min_periods=1).mean()

In [35]:
if "liquidity_index" not in df_crypto.columns:
    raise ValueError("liquidity_index missing — cannot compute CMI")

In [36]:
df_crypto["CMI_raw"] = (
    0.45 * df_crypto["btc_signal"] +
    0.35 * df_crypto["eth_signal"] +
    0.20 * df_crypto["liquidity_index"]
)

In [37]:
print(df_crypto.columns)

Index(['date', 'bitcoin_price_usd', 'bitcoin_volume_usd', 'ethereum_price_usd',
       'ethereum_volume_usd', 'tether_price_usd', 'tether_volume_usd',
       'market_liquidity', 'liquidity_index', 'btc_return', 'eth_return',
       'btc_signal', 'eth_signal', 'CMI', 'CMI_index', 'CMI_raw'],
      dtype='object')


In [38]:
import numpy as np

# -----------------------------
# 1. Compute returns (if not already reliable)
# -----------------------------
df_crypto["btc_return"] = df_crypto["bitcoin_price_usd"].pct_change().fillna(0)
df_crypto["eth_return"] = df_crypto["ethereum_price_usd"].pct_change().fillna(0)

# -----------------------------
# 2. Rolling volatility (7-day window)
#    → measures market instability
# -----------------------------
df_crypto["btc_volatility"] = df_crypto["btc_return"].rolling(7, min_periods=1).std()
df_crypto["eth_volatility"] = df_crypto["eth_return"].rolling(7, min_periods=1).std()

# -----------------------------
# 3. Liquidity stress (inverse signal)
#    → lower liquidity = higher stress
# -----------------------------
df_crypto["liquidity_stress"] = 1 / (df_crypto["liquidity_index"] + 1e-6)

# -----------------------------
# 4. Normalize components (important for fair weighting)
# -----------------------------
df_crypto["btc_vol_norm"] = df_crypto["btc_volatility"] / df_crypto["btc_volatility"].max()
df_crypto["eth_vol_norm"] = df_crypto["eth_volatility"] / df_crypto["eth_volatility"].max()
df_crypto["liq_stress_norm"] = df_crypto["liquidity_stress"] / df_crypto["liquidity_stress"].max()

# -----------------------------
# 5. Market Stress Index (MSI)
# -----------------------------
df_crypto["MSI"] = (
    0.4 * df_crypto["btc_vol_norm"] +
    0.3 * df_crypto["eth_vol_norm"] +
    0.3 * df_crypto["liq_stress_norm"]
)

# -----------------------------
# 6. Smooth final signal
# -----------------------------
df_crypto["MSI"] = df_crypto["MSI"].rolling(3, min_periods=1).mean()

# -----------------------------
# 7. Preview
# -----------------------------
print(df_crypto[["date", "MSI"]].tail(10))

           date       MSI
355  2026-06-11  0.496309
356  2026-06-12  0.470370
357  2026-06-13  0.413235
358  2026-06-14  0.395041
359  2026-06-15  0.342474
360  2026-06-16  0.320295
361  2026-06-17  0.266282
362  2026-06-18  0.252190
363  2026-06-19  0.262310
364  2026-06-20  0.284830


In [39]:
import numpy as np
import pandas as pd

# =========================
# 1. DATE STANDARDIZATION
# =========================
df_crypto["date"] = pd.to_datetime(df_crypto["date"])

# WHY:
# We enforce datetime format so we can:
# - sort time series correctly
# - compute rolling indicators
# - align multi-asset signals

# =========================
# 2. DAILY RETURNS (MARKET DIRECTION SIGNAL)
# =========================

df_crypto["btc_return"] = df_crypto["bitcoin_price_usd"].pct_change().fillna(0)
df_crypto["eth_return"] = df_crypto["ethereum_price_usd"].pct_change().fillna(0)

# WHY:
# Returns measure DAILY MARKET DIRECTION (up/down movement).
# - Positive return = bullish momentum
# - Negative return = bearish pressure
# USED FOR:
# → Detecting trend direction
# → Feeding momentum into CMI (market strength)

# =========================
# 3. SMOOTHED SIGNALS (NOISE REDUCTION)
# =========================

df_crypto["btc_signal"] = df_crypto["btc_return"].rolling(3, min_periods=1).mean()
df_crypto["eth_signal"] = df_crypto["eth_return"].rolling(3, min_periods=1).mean()

# WHY:
# Crypto is extremely noisy day-to-day.
# Smoothing helps us:
# - filter fake spikes
# - capture short-term trend direction instead of noise
#
# USED FOR DECISION:
# → If smoothed signal > 0: sustained bullish trend
# → If smoothed signal < 0: sustained bearish trend

# =========================
# 4. LIQUIDITY INDEX (MARKET PARTICIPATION HEALTH)
# =========================

df_crypto["liquidity_index"] = df_crypto["liquidity_index"].fillna(method="ffill")

# WHY:
# Liquidity measures how easily assets are traded without price distortion.
#
# INTERPRETATION:
# - High liquidity = stable, healthy market (low manipulation risk)
# - Low liquidity = fragile market (high slippage, manipulation risk)
#
# USED FOR DECISION:
# → Low liquidity + high volatility = dangerous trading environment

# =========================
# 5. CMI (CRYPTO MARKET INDEX - OVERALL MARKET STRENGTH)
# =========================

df_crypto["CMI_raw"] = (
    0.45 * df_crypto["btc_signal"] +
    0.35 * df_crypto["eth_signal"] +
    0.20 * df_crypto["liquidity_index"]
)

# WHY:
# CMI combines:
# - BTC (market leader momentum)
# - ETH (secondary market confirmation)
# - liquidity (market health)
#
# INTERPRETATION:
# → High CMI = strong bullish market conditions
# → Low CMI = weak or bearish market environment
#
# USED FOR DECISION:
# → Helps decide “risk-on vs risk-off” market stance

# Fill missing safely
df_crypto["CMI_raw"] = df_crypto["CMI_raw"].fillna(method="ffill").fillna(0)

# =========================
# 6. CMI INDEX (NORMALIZED MARKET STRENGTH SCORE)
# =========================

cmi_min = df_crypto["CMI_raw"].min()
cmi_max = df_crypto["CMI_raw"].max()

df_crypto["CMI_index"] = 0 if cmi_min == cmi_max else (
    (df_crypto["CMI_raw"] - cmi_min) / (cmi_max - cmi_min)
) * 100

# WHY:
# Raw values are hard to interpret.
# Normalization converts CMI into a 0–100 score.
#
# INTERPRETATION:
# - 0–30 → weak market (bearish)
# - 30–70 → neutral / sideways
# - 70–100 → strong bullish conditions
#
# USED FOR DECISION:
# → Entry timing filter
# → Market regime classification

# =========================
# 7. MARKET STRESS INDEX (MSI - RISK / FEAR GAUGE)
# =========================

df_crypto["btc_volatility"] = df_crypto["btc_return"].rolling(7).std()
df_crypto["eth_volatility"] = df_crypto["eth_return"].rolling(7).std()

# WHY:
# Volatility measures uncertainty and instability.
#
# INTERPRETATION:
# - High volatility = panic / fear / instability
# - Low volatility = calm / stable market
#
# USED FOR DECISION:
# → Avoid trading during extreme volatility spikes

# Liquidity stress (inverse relationship)
df_crypto["liquidity_stress"] = 1 / (df_crypto["liquidity_index"] + 1e-6)

# WHY:
# When liquidity drops, markets become unstable.
# We invert liquidity to reflect stress.
#
# INTERPRETATION:
# - High liquidity stress = fragile / manipulated market
# - Low liquidity stress = stable market

# =========================
# MSI RAW (COMBINED MARKET FEAR SIGNAL)
# =========================

df_crypto["MSI_raw"] = (
    0.4 * df_crypto["btc_volatility"] +
    0.4 * df_crypto["eth_volatility"] +
    0.2 * df_crypto["liquidity_stress"]
)

# WHY:
# MSI measures TOTAL MARKET RISK combining:
# - price instability (volatility)
# - liquidity breakdown (stress)
#
# INTERPRETATION:
# → High MSI = fear, instability, crash risk
# → Low MSI = stable, calm market
#
# USED FOR DECISION:
# → Risk management filter (DO NOT trade aggressively when MSI is high)

df_crypto["MSI_raw"] = df_crypto["MSI_raw"].fillna(method="bfill").fillna(0)

# =========================
# 8. MSI INDEX (STANDARDIZED RISK SCORE)
# =========================

msi_min = df_crypto["MSI_raw"].min()
msi_max = df_crypto["MSI_raw"].max()

df_crypto["MSI_index"] = 0 if msi_min == msi_max else (
    (df_crypto["MSI_raw"] - msi_min) / (msi_max - msi_min)
) * 100

# WHY:
# Converts risk into an intuitive scale (0–100)
#
# INTERPRETATION:
# - 0–30 → low risk (safe environment)
# - 30–70 → moderate risk
# - 70–100 → extreme risk (avoid exposure)
#
# USED FOR DECISION:
# → Position sizing (smaller when MSI high)
# → Entry/exit timing filter

# =========================
# 9. FINAL ANALYTICS TABLE
# =========================

final_cols = [
    "date",

    # Raw market data
    "bitcoin_price_usd",
    "ethereum_price_usd",
    "bitcoin_volume_usd",
    "ethereum_volume_usd",

    # Market direction
    "btc_return",
    "eth_return",
    "btc_signal",
    "eth_signal",

    # Market health
    "liquidity_index",

    # Market strength (trend)
    "CMI_raw",
    "CMI_index",

    # Market risk (fear/stress)
    "MSI_raw",
    "MSI_index"
]

print(df_crypto[final_cols].head(10))

        date  bitcoin_price_usd  ethereum_price_usd  bitcoin_volume_usd  \
0 2025-06-21      103290.105145         2405.695434        3.063275e+10   
1 2025-06-22      101532.568385         2270.581996        2.164446e+10   
2 2025-06-23      100852.582646         2227.428727        5.128714e+10   
3 2025-06-24      105511.624379         2423.899644        5.001180e+10   
4 2025-06-25      105976.069298         2446.539302        3.170464e+10   
5 2025-06-26      107238.530450         2417.226786        3.046827e+10   
6 2025-06-27      106984.012538         2415.031865        2.432070e+10   
7 2025-06-28      107078.915606         2423.030290        2.519431e+10   
8 2025-06-29      107331.585485         2437.133002        8.960878e+09   
9 2025-06-30      108396.616313         2502.667272        1.488108e+10   

   ethereum_volume_usd  btc_return  eth_return  btc_signal  eth_signal  \
0         2.048282e+10    0.000000    0.000000    0.000000    0.000000   
1         1.615561e+10   -

C:\Users\User\AppData\Local\Temp\ipykernel_13480\22784935.py:51: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_crypto["liquidity_index"] = df_crypto["liquidity_index"].fillna(method="ffill")
C:\Users\User\AppData\Local\Temp\ipykernel_13480\22784935.py:87: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_crypto["CMI_raw"] = df_crypto["CMI_raw"].fillna(method="ffill").fillna(0)
C:\Users\User\AppData\Local\Temp\ipykernel_13480\22784935.py:163: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_crypto["MSI_raw"] = df_crypto["MSI_raw"].fillna(method="bfill").fillna(0)


In [42]:
# Output path relative to notebooks/ folder
output_path = "../data/crypto_market_data.csv"

# Save cleaned dataset to CSV (final analysis-ready dataset)
df_crypto.to_csv(output_path, index=False)

# Read back file to confirm it was saved correctly
df_verify = pd.read_csv(output_path)

# Basic verification outputs
print("Saved successfully!")
print("File:", output_path)
print("Shape:", df_verify.shape)
print("Columns:", df_verify.columns.tolist())

# Preview first rows (data integrity check)
print("\nFirst 3 rows:")
print(df_verify.head(3))

# Preview last rows (check continuity and completeness)
print("\nLast 3 rows:")
print(df_verify.tail(3))

Saved successfully!
File: ../data/crypto_market_data.csv
Shape: (365, 25)
Columns: ['date', 'bitcoin_price_usd', 'bitcoin_volume_usd', 'ethereum_price_usd', 'ethereum_volume_usd', 'tether_price_usd', 'tether_volume_usd', 'market_liquidity', 'liquidity_index', 'btc_return', 'eth_return', 'btc_signal', 'eth_signal', 'CMI', 'CMI_index', 'CMI_raw', 'btc_volatility', 'eth_volatility', 'liquidity_stress', 'btc_vol_norm', 'eth_vol_norm', 'liq_stress_norm', 'MSI', 'MSI_raw', 'MSI_index']

First 3 rows:
         date  bitcoin_price_usd  bitcoin_volume_usd  ethereum_price_usd  \
0  2025-06-21      103290.105145        3.063275e+10         2405.695434   
1  2025-06-22      101532.568385        2.164446e+10         2270.581996   
2  2025-06-23      100852.582646        5.128714e+10         2227.428727   

   ethereum_volume_usd  tether_price_usd  tether_volume_usd  market_liquidity  \
0         2.048282e+10          1.000017       2.772718e+10      7.884275e+10   
1         1.615561e+10          1

In [43]:
# Cell 9 — Data Quality Check (run before saving or pushing to GitHub)

print("=== DATA QUALITY REPORT ===")

# Basic structure checks
print(f"\nRows: {len(df_crypto)}")
print(f"Columns: {len(df_crypto.columns)}")

# Missing values check (critical for model reliability)
total_nulls = df_crypto.isnull().sum().sum()
print(f"Nulls: {total_nulls} total")

# Date coverage check (ensures time continuity)
print("\nDate range:")
print(f" Start: {df_crypto['date'].min()}")
print(f" End: {df_crypto['date'].max()}")

# BTC sanity check (detect data corruption or API issues)
print("\nBTC price range:")
print(f" Min: ${df_crypto['bitcoin_price_usd'].min():,.2f}")
print(f" Max: ${df_crypto['bitcoin_price_usd'].max():,.2f}")

# ETH sanity check
print("\nETH price range:")
print(f" Min: ${df_crypto['ethereum_price_usd'].min():,.2f}")
print(f" Max: ${df_crypto['ethereum_price_usd'].max():,.2f}")

# Data type validation (ensures feature engineering consistency)
print("\nData types:")
print(df_crypto.dtypes)

# =========================
# HARD STOP RULE (QUALITY GATE)
# =========================
if total_nulls > 0:
    print("\n⚠ WARNING: Dataset contains null values. Fix before committing.")

if df_crypto["bitcoin_price_usd"].min() <= 0:
    print("\n⚠ WARNING: BTC price has invalid values (<= 0). Check data source.")

if df_crypto["ethereum_price_usd"].min() <= 0:
    print("\n⚠ WARNING: ETH price has invalid values (<= 0). Check data source.")
    

=== DATA QUALITY REPORT ===

Rows: 365
Columns: 25
Nulls: 15 total

Date range:
 Start: 2025-06-21 00:00:00
 End: 2026-06-20 00:00:00

BTC price range:
 Min: $60,861.88
 Max: $124,773.51

ETH price range:
 Min: $1,568.77
 Max: $4,829.23

Data types:
date                   datetime64[ns]
bitcoin_price_usd             float64
bitcoin_volume_usd            float64
ethereum_price_usd            float64
ethereum_volume_usd           float64
tether_price_usd              float64
tether_volume_usd             float64
market_liquidity              float64
liquidity_index               float64
btc_return                    float64
eth_return                    float64
btc_signal                    float64
eth_signal                    float64
CMI                           float64
CMI_index                     float64
CMI_raw                       float64
btc_volatility                float64
eth_volatility                float64
liquidity_stress              float64
btc_vol_norm                

In [44]:
df_crypto.to_csv("crypto_market_data.csv", index=False)

In [45]:
import pandas as pd

df_check = pd.read_csv("WEEK 1/data/crypto_market_data.csv")

print(df_check.shape)
print(df_check.head())

FileNotFoundError: [Errno 2] No such file or directory: 'WEEK 1/data/crypto_market_data.csv'

In [46]:
df_crypto.to_csv("WEEK 1/data/crypto_market_data.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'WEEK 1\data'